## Packages

In [1]:
import pandas as pd
import numpy as np

from PIL import Image, ImageOps
Image.init()

1

## CONTEXTE

In [2]:
DATA_JDD = "/workspace/data/jdd/ImageAndXML_Data.zip"
JSON_IMAGES = "/workspace/data/raw/ImageAndXML_Data_Images.json"
JSON_GT = "/workspace/data/raw/ImageAndXML_Data_GT.json"
JSON_OCR = "/workspace/data/raw/ImageAndXML_Data_OCR.json"

IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg", ".tif", ".tiff")


## LOAD DATA

In [3]:
import xml.etree.ElementTree as ET
from pathlib import Path
import os
import base64
from  io import BytesIO
from ipywidgets import IntProgress

# UNZIP  in Cell
import zipfile

z = zipfile.ZipFile(DATA_JDD, "r")

# IMAGES
def load_image_to_base64(z: zipfile.ZipFile, image_path):
    with z.open(image_path) as f:
        data = f.read()
        return  base64.b64encode(data)

def load_images_to_dataframe(z: zipfile.ZipFile) -> pd.DataFrame:
    image_paths = []
    image_files = []
    image_types = []
    images_base64 = []
    widths = []
    heights = []

    progress = IntProgress(
        value=0,
        min=0,
        max=len(z.namelist()),
        description='Load images',
        bar_style='info',
        style={'description_width': '200px'}
    )
    display(progress)
    for f in z.namelist():
        progress.value += 1
        progress.description= os.path.basename(f) + ' is loading...'
        if f.lower().endswith(IMAGE_EXTENSIONS):
            img = load_image_to_base64(z, f)
            image_paths.append(f)
            image_files.append(os.path.basename(f))
            image_types.append(os.path.splitext(f)[1].lower().replace(".", ""))
            images_base64.append(img)
    
    progress.layout.display = 'none'
        
    
    df = pd.DataFrame({ 
        "image_filename": image_files,
        "image_type": image_types,
        "image_base64": images_base64
    })
    
    return df


print("chargement dataframe des images")
df_IMG = load_images_to_dataframe(z)

print("DataFrame df_IMG -> JSON ")
df_IMG.to_json(JSON_IMAGES, orient="records", indent=2)

# GT XML
def load_gtxml_to_dataframe(f: str, z: zipfile.ZipFile) -> pd.DataFrame:	
    # Charger le XML GT
    xml_bytes = z.read(f)
    root = ET.fromstring(xml_bytes)

    # Namespace PAGE-XML
    ns = {
        "pc": "http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15"
    }

    rows = []

    # Récupérer la page
    page = root.find(".//pc:Page", ns)
    image_filename = page.get("imageFilename")
    image_width = int(page.get("imageWidth"))
    image_height = int(page.get("imageHeight"))

    # Parcourir les TextRegion
    for region in page.findall("pc:TextRegion", ns):
        region_id = region.get("id")

        # Propriété "entity"
        prop = region.find("pc:Property[@key='entity']", ns)
        entity = prop.get("value") if prop is not None else None

        # Coordonnées polygonales
        coords = region.find("pc:Coords", ns)
        points = coords.get("points") if coords is not None else None

        rows.append({
            "region_id": region_id,
            "entity": entity,
            "points": points,
            "image_filename": image_filename,
            "image_width": image_width,
            "image_height": image_height
        })
    return pd.DataFrame(rows)

print("chargement dataframe GT XML")
df_GT_XML = pd.concat([ load_gtxml_to_dataframe(f, z) for f in z.namelist() if f.lower().endswith("_gt.xml")])

print("DataFrame df_GT_XML -> JSON")
df_GT_XML.to_json(JSON_GT, orient="records", indent=2)

# OCR XML
def load_ocrxml_to_dataframe(f: str, z: zipfile.ZipFile) -> pd.DataFrame:

    # Charger le XML OCR
    xml_bytes = z.read(f)
    root = ET.fromstring(xml_bytes)

    # Namespace PAGE-XML
    ns = {"pc": "http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15"}

    rows = []

    # Récupérer les infos de la page
    page = root.find(".//pc:Page", ns)
    image_filename = page.get("imageFilename")
    image_width = int(page.get("imageWidth"))
    image_height = int(page.get("imageHeight"))

    # Parcourir toutes les TextRegion
    for region in page.findall("pc:TextRegion", ns):
        region_id = region.get("id")

        # Parcourir les lignes
        for line in region.findall("pc:TextLine", ns):
            line_id = line.get("id")

            # Parcourir les mots
            for word in line.findall("pc:Word", ns):
                word_id = word.get("id")

                # Coordonnées polygonales
                coords = word.find("pc:Coords", ns)
                points = coords.get("points") if coords is not None else None

                # Texte OCR
                unicode_el = word.find("pc:TextEquiv/pc:Unicode", ns)
                text = unicode_el.text if unicode_el is not None else ""

                # Score de confiance
                conf_el = word.find("pc:TextEquiv", ns)
                conf = float(conf_el.get("conf")) if conf_el is not None else None

                rows.append({
                    "region_id": region_id,
                    "line_id": line_id,
                    "word_id": word_id,
                    "text": text,
                    "confidence": conf,
                    "points": points,
                    "image_filename": image_filename,
                    "image_width": image_width,
                    "image_height": image_height
                })
    return pd.DataFrame(rows)

print("chargement dataframe OCR XML")
df_OCR_XML = pd.concat([load_ocrxml_to_dataframe(f, z) for f in z.namelist() if f.lower().endswith("_ocr.xml")])

print("DataFrame df_OCR_XML -> JSON")
df_OCR_XML.to_json(JSON_OCR, orient="records", indent=2)


chargement dataframe des images


IntProgress(value=0, bar_style='info', description='Load images', max=1560, style=ProgressStyle(description_wi…

DataFrame df_IMG -> JSON 
chargement dataframe GT XML
DataFrame df_GT_XML -> JSON
chargement dataframe OCR XML
DataFrame df_OCR_XML -> JSON
